# AdverTest GPU Worker — yolov7-face
Decoupled heavy-compute stage: **triage → attack synthesis → retrain → blind evaluation gate**.
Runtime: GPU (T4 or better). Raw data arrives through Data Block presigned URLs and is deleted at the end of the run.

In [ ]:
!pip -q install albumentations opencv-python-headless numpy pyyaml torch torchvision pytorch-fid
!git clone -q https://github.com/derronqi/yolov7-face.git third_party/yolov7_face || true
import os, sys, json, glob, shutil, time
sys.path.insert(0, 'third_party/yolov7_face')
CFG = dict(
    weights='weights/yolov7-face.pt',
    wider_val_url=os.environ.get('WIDER_VAL_PRESIGNED_URL', ''),   # Base domain (clean)
    ego_val_url=os.environ.get('EGO4D_VAL_PRESIGNED_URL', ''),     # Target domain (egocentric)
    iou_thr=0.5, conf_thr=0.4,            # hard-negative definition
    max_base_drop=0.01,                   # gate: base mAP may drop at most 1%
    min_corrupted_gain=0.05,              # gate: corrupted mAP must improve meaningfully
)
os.makedirs('work', exist_ok=True)

## 1 · Ingest (presigned URL, TTL-aware, temporary cache)

In [ ]:
import urllib.request, urllib.error, time
from urllib.parse import urlparse, parse_qs

def fetch(url, dest, retries=3, timeout=30):
    q = parse_qs(urlparse(url).query)
    if 'Expires' in q and int(q['Expires'][0]) < time.time():
        raise RuntimeError('Presigned URL expired — request a fresh one from the Data Block platform.')
    for i in range(retries + 1):
        try:
            urllib.request.urlretrieve(url, dest); return dest
        except (urllib.error.URLError, TimeoutError) as e:
            if i == retries: raise
            time.sleep(2 ** i)

for name, url in [('wider_val.zip', CFG['wider_val_url']), ('ego_val.zip', CFG['ego_val_url'])]:
    if url:
        fetch(url, f'work/{name}'); shutil.unpack_archive(f'work/{name}', f'work/{name[:-4]}')
    else:
        print(f'[skip] no presigned URL for {name} — set the env var to run on real data')

## 2 · Smart triage — isolate hard negatives only
Hard negative = best IoU < 0.5 **or** confidence < 0.4. Only this subset leaves the worker.

In [ ]:
import numpy as np

def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1]); x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua else 0.0

def is_hard_negative(gt_boxes, preds):
    """preds: list of (x1,y1,x2,y2,conf). True if any GT face is missed / weak."""
    for g in gt_boxes:
        best = max(preds, key=lambda p: iou(g, p[:4]), default=None)
        if best is None or iou(g, best[:4]) < CFG['iou_thr'] or best[4] < CFG['conf_thr']:
            return True
    return False

assert is_hard_negative([(0,0,10,10)], []) and not is_hard_negative([(0,0,10,10)], [(0,0,10,10,0.9)])
# TODO(run): loop over work/ego_val with detect.py outputs and write work/hard_negatives.json

## 3 · Constraint-driven attack synthesis (with audit trail)
Exclusion matrix, bbox-integrity gate (>50% loss ⇒ discard), biomechanical IMU blur at 2–5 Hz.

In [ ]:
import cv2, math

CONFLICTS = [{'flare', 'low_light'}, {'flare', 'high_iso'}, {'flare', 'heavy_rain'}]
def exclusion_ok(tags):
    return not any(c <= set(tags) for c in CONFLICTS)

def imu_kernel(severity=3, hz=3.0, n=200):
    k = 5 + severity * 10; ker = np.zeros((k, k), np.float32); c = k // 2; amp = k * 0.4
    for i in range(n):
        t = i / n
        x = c + (t - .5) * amp
        y = c + amp * .3 * math.sin(2 * math.pi * (hz / 2) * t) + amp * .15 * (t - .5) ** 3 * 8
        xi, yi = int(round(x)), int(round(y))
        if 0 <= xi < k and 0 <= yi < k: ker[yi, xi] += 1
    ker = cv2.GaussianBlur(ker, (3, 3), 0); return ker / ker.sum()

def bbox_ok(box, w, h, max_loss=0.5):
    x1, y1, x2, y2 = box; area = (x2-x1)*(y2-y1)
    inside = max(0, min(x2, w)-max(x1, 0)) * max(0, min(y2, h)-max(y1, 0))
    return area > 0 and inside / area >= 1 - max_loss

def synthesize(img, box, tag, severity, hz):
    assert exclusion_ok([tag]) and bbox_ok(box, *img.shape[:2][::-1])
    out = cv2.filter2D(img, -1, imu_kernel(severity, hz))
    audit = {'insight_source': tag, 'applied_chain': ['BiomechanicalBlur'], 'severity_level': severity,
             'justification': f'Baseline conf<0.4 under head-bob; IMU blur at {hz} Hz selected at severity {severity}.'}
    return out, audit

## 4 · Retrain (fine-tune on hard negatives + replay of clean WIDER FACE)
Mixing a clean replay buffer is what protects the base domain from catastrophic forgetting.

In [ ]:
# Example (run inside the cloned yolov7-face repo, after writing data/advertest.yaml):
# !python train.py --weights {CFG['weights']} --data data/advertest.yaml --epochs 30 --batch-size 16 --img 640 --hyp data/hyp.scratch.yaml --name advertest_v2
print('Training command documented above; skipped in dry-run mode.')

## 5 · Blind evaluation gate

In [ ]:
def gate(base_v1, base_v2, corr_v1, corr_v2):
    drop, gain = base_v1 - base_v2, corr_v2 - corr_v1
    ok = drop <= CFG['max_base_drop'] + 1e-9 and gain >= CFG['min_corrupted_gain']
    return {'accepted': ok, 'base_drop': round(drop, 4), 'corrupted_gain': round(gain, 4)}

print(gate(0.85, 0.84, 0.41, 0.68))    # accepted
print(gate(0.85, 0.80, 0.41, 0.70))    # rejected: base mAP dropped > 1%

## 6 · Phase-2 GenAI bridge package + FID validation, then wipe raw data

In [ ]:
def package_for_genai(items, out='work/genai_bridge'):
    os.makedirs(out, exist_ok=True)
    manifest = [{'image': i['image'], 'bbox': i['bbox'], 'landmarks': i.get('landmarks'), 'audit_trail': i['audit']} for i in items]
    json.dump(manifest, open(f'{out}/manifest.json', 'w'), indent=2); return out

# Validate returned images:  !python -m pytorch_fid work/genai_returned work/real_reference --device cuda
# Accept only if FID <= threshold agreed with the Data Platform team.

def wipe_raw():
    for d in ('work/wider_val', 'work/ego_val', 'work/wider_val.zip', 'work/ego_val.zip'):
        shutil.rmtree(d, ignore_errors=True) if os.path.isdir(d) else (os.path.exists(d) and os.remove(d))
wipe_raw()